# KUL CV GA2 - Image Classification Training on Kaggle GPU

Self-contained notebook for Section 2.1 image classification. Run all cells top-to-bottom on Kaggle with GPU enabled.

Outputs are grouped by experiment:

```text
/kaggle/working/image_classification/efficientnet_b3_320/
  checkpoints/
  metrics/
  predictions/
  submissions/
```

The Kaggle submission CSV is named `submission_classification_efficientnet_b3_320.csv`.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# Config
DATA_DIR = Path('/kaggle/input/competitions/kul-computer-vision-ga-2-2026')
EXPERIMENT_NAME = 'efficientnet_b3_320'
OUTPUT_DIR = Path('/kaggle/working/image_classification') / EXPERIMENT_NAME
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
METRICS_DIR = OUTPUT_DIR / 'metrics'
PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
SUBMISSIONS_DIR = OUTPUT_DIR / 'submissions'

for directory in (CHECKPOINT_DIR, METRICS_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LABELS = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

BACKBONE = 'efficientnet_b3'
IMG_SIZE = 320
BATCH_SIZE = 32
NUM_WORKERS = 4
VAL_SPLIT = 0.2
RANDOM_SEED = 42
WEIGHT_DECAY = 1e-4

STAGE1_EPOCHS, STAGE1_LR = 5, 1e-3
STAGE2_EPOCHS, STAGE2_LR = 20, 1e-4
STAGE3_EPOCHS, STAGE3_LR = 5, 5e-5

BEST_CKPT = CHECKPOINT_DIR / f'best_model_{EXPERIMENT_NAME}.pth'
LAST_CKPT = CHECKPOINT_DIR / f'last_model_{EXPERIMENT_NAME}.pth'
FINAL_CKPT = CHECKPOINT_DIR / f'final_model_{EXPERIMENT_NAME}.pth'
THRESHOLDS_PATH = METRICS_DIR / f'best_thresholds_{EXPERIMENT_NAME}.npy'
THRESHOLDS_CSV = METRICS_DIR / f'thresholds_{EXPERIMENT_NAME}.csv'
SUMMARY_CSV = METRICS_DIR / f'evaluation_summary_{EXPERIMENT_NAME}.csv'
HISTORY_CSV = METRICS_DIR / f'training_history_{EXPERIMENT_NAME}.csv'
PROBABILITY_CSV = PREDICTIONS_DIR / f'test_probabilities_{EXPERIMENT_NAME}.csv'
BINARY_PREDICTION_CSV = PREDICTIONS_DIR / f'test_binary_predictions_{EXPERIMENT_NAME}.csv'
SUBMISSION_CSV = SUBMISSIONS_DIR / f'submission_classification_{EXPERIMENT_NAME}.csv'

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print('Experiment:', EXPERIMENT_NAME)
print('Output:', OUTPUT_DIR)


In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs_neg = 1.0 - probs
        if self.clip > 0:
            probs_neg = (probs_neg + self.clip).clamp(max=1.0)

        log_p = torch.log(probs.clamp(min=self.eps))
        log_np = torch.log(probs_neg.clamp(min=self.eps))
        loss = targets * log_p + (1 - targets) * log_np

        with torch.no_grad():
            weights = (
                targets * (1 - probs).pow(self.gamma_pos)
                + (1 - targets) * probs.pow(self.gamma_neg)
            )
        return -(loss * weights).mean()


In [ ]:
_MEAN = [0.485, 0.456, 0.406]
_STD = [0.229, 0.224, 0.225]


def get_train_transform(img_size=320):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.RandomHorizontalFlip(),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
        T.RandomErasing(p=0.3, scale=(0.02, 0.2)),
    ])


def get_val_transform(img_size=320):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])


class VOCDataset(Dataset):
    def __init__(self, df, data_dir, split='train', transform=None):
        self.df = df
        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform or get_val_transform()
        self.has_labels = all(column in df.columns for column in LABELS)
        self.indices = list(df.index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        arr = np.load(self.data_dir / self.split / 'img' / f'{self.split}_{idx}.npy')
        img = self.transform(Image.fromarray(arr))
        if self.has_labels:
            label = torch.FloatTensor(self.df.loc[idx, LABELS].values.astype(float))
            return img, label
        return img, idx


def load_train_df():
    return pd.read_csv(DATA_DIR / 'train' / 'train_set.csv', index_col='Id')


def load_test_df():
    return pd.read_csv(DATA_DIR / 'test' / 'test_set.csv', index_col='Id')


In [ ]:
class MultiLabelClassifier(nn.Module):
    def __init__(self, backbone='efficientnet_b3', num_classes=20, pretrained=True):
        super().__init__()
        if backbone == 'efficientnet_b3':
            model = models.efficientnet_b3(
                weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
            )
            self.features = nn.Sequential(model.features, model.avgpool)
            feat_dim = 1536
        elif backbone == 'resnet50':
            model = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
            )
            self.features = nn.Sequential(*list(model.children())[:-1])
            feat_dim = 2048
        else:
            raise ValueError(f'Unknown backbone: {backbone}')

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes),
        )
        for layer in self.classifier.modules():
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def freeze_backbone(self):
        for parameter in self.features.parameters():
            parameter.requires_grad = False

    def unfreeze_backbone(self):
        for parameter in self.features.parameters():
            parameter.requires_grad = True

    def trainable_params(self):
        return sum(parameter.numel() for parameter in self.parameters() if parameter.requires_grad)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))


In [ ]:
def run_epoch(model, loader, criterion, optimizer, train):
    model.train(train)
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False, desc='train' if train else 'val'):
            imgs = imgs.to(device)
            labels = labels.to(device)
            loss = criterion(model(imgs), labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(imgs)
    return total_loss / len(loader.dataset)


def train_stage(model, train_loader, val_loader, criterion, lr, epochs, stage_name, best_val_loss, history):
    optimizer = torch.optim.AdamW(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss = run_epoch(model, val_loader, criterion, optimizer, train=False)
        scheduler.step()

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss
            torch.save(model.state_dict(), BEST_CKPT)
        torch.save(model.state_dict(), LAST_CKPT)

        history.append({
            'experiment': EXPERIMENT_NAME,
            'stage': stage_name,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'lr': lr,
            'is_best': is_best,
        })

        flag = ' <- best' if is_best else ''
        print(f'[{stage_name}] Epoch {epoch:3d}/{epochs} | train={train_loss:.4f} val={val_loss:.4f}{flag}')

    return best_val_loss


In [ ]:
df = load_train_df()
train_idx, val_idx = train_test_split(
    range(len(df)),
    test_size=VAL_SPLIT,
    random_state=RANDOM_SEED,
)

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    VOCDataset(df.iloc[train_idx], DATA_DIR, split='train', transform=get_train_transform(IMG_SIZE)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    VOCDataset(df.iloc[val_idx], DATA_DIR, split='train', transform=get_val_transform(IMG_SIZE)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

model = MultiLabelClassifier(backbone=BACKBONE, num_classes=len(LABELS), pretrained=True).to(device)
criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05)
history = []
best_loss = float('inf')

print('\n=== Stage 1: train classification head only ===')
model.freeze_backbone()
print(f'Trainable params: {model.trainable_params():,}')
best_loss = train_stage(
    model, train_loader, val_loader, criterion,
    STAGE1_LR, STAGE1_EPOCHS, 'S1', best_loss, history,
)

print('\n=== Stage 2: full fine-tuning ===')
model.unfreeze_backbone()
print(f'Trainable params: {model.trainable_params():,}')
best_loss = train_stage(
    model, train_loader, val_loader, criterion,
    STAGE2_LR, STAGE2_EPOCHS, 'S2', best_loss, history,
)

print(f'\n=== Stage 3: retrain on all {len(df)} training samples ===')
model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
full_loader = DataLoader(
    VOCDataset(df, DATA_DIR, split='train', transform=get_train_transform(IMG_SIZE)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
optimizer3 = torch.optim.AdamW(model.parameters(), lr=STAGE3_LR, weight_decay=WEIGHT_DECAY)
scheduler3 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer3, T_max=STAGE3_EPOCHS)
for epoch in range(1, STAGE3_EPOCHS + 1):
    loss = run_epoch(model, full_loader, criterion, optimizer3, train=True)
    scheduler3.step()
    history.append({
        'experiment': EXPERIMENT_NAME,
        'stage': 'S3',
        'epoch': epoch,
        'train_loss': loss,
        'val_loss': np.nan,
        'lr': STAGE3_LR,
        'is_best': False,
    })
    print(f'[S3] Epoch {epoch}/{STAGE3_EPOCHS} | loss={loss:.4f}')

torch.save(model.state_dict(), FINAL_CKPT)
pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)
print(f'\nDone. Best val loss: {best_loss:.4f}')
print('Saved checkpoints:')
print(' ', BEST_CKPT)
print(' ', LAST_CKPT)
print(' ', FINAL_CKPT)
print('Saved history:', HISTORY_CSV)


In [ ]:
model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Val inference'):
        probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

map_score = average_precision_score(all_labels, all_probs, average='macro')
print(f'Val mAP: {map_score:.4f}')

threshold_rows = []
thresholds = np.zeros(len(LABELS))
for i, class_name in enumerate(LABELS):
    class_ap = average_precision_score(all_labels[:, i], all_probs[:, i])
    best_t, best_f1 = 0.5, 0.0
    for threshold in np.arange(0.1, 0.91, 0.05):
        preds = (all_probs[:, i] > threshold).astype(int)
        tp = (preds * all_labels[:, i]).sum()
        fp = (preds * (1 - all_labels[:, i])).sum()
        fn = ((1 - preds) * all_labels[:, i]).sum()
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
        if f1 > best_f1:
            best_f1, best_t = f1, threshold
    thresholds[i] = best_t
    threshold_rows.append({
        'class': class_name,
        'ap': class_ap,
        'best_threshold': best_t,
        'best_f1': best_f1,
    })
    print(f'  {class_name:<14s} AP={class_ap:.4f} threshold={best_t:.2f} F1={best_f1:.3f}')

np.save(THRESHOLDS_PATH, thresholds)
pd.DataFrame(threshold_rows).to_csv(THRESHOLDS_CSV, index=False)
pd.DataFrame([{
    'experiment': EXPERIMENT_NAME,
    'backbone': BACKBONE,
    'img_size': IMG_SIZE,
    'mAP': map_score,
    'checkpoint': BEST_CKPT.name,
}]).to_csv(SUMMARY_CSV, index=False)

print('Saved thresholds:', THRESHOLDS_PATH)
print('Saved threshold CSV:', THRESHOLDS_CSV)
print('Saved summary:', SUMMARY_CSV)


In [ ]:
def rle_encode(arr):
    pixels = np.concatenate([[0], arr.flatten(), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


ckpt = FINAL_CKPT if FINAL_CKPT.exists() else BEST_CKPT
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()
print('Using checkpoint:', ckpt)

if THRESHOLDS_PATH.exists():
    thresholds = np.load(THRESHOLDS_PATH)
else:
    thresholds = np.full(len(LABELS), 0.5)
    print('Threshold file not found; using 0.5 for every class.')

test_df = load_test_df()
test_ds = VOCDataset(test_df, DATA_DIR, split='test', transform=get_val_transform(IMG_SIZE))
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

all_probs = []
with torch.no_grad():
    for imgs, _ in tqdm(test_loader, desc='Test inference (TTA)'):
        imgs = imgs.to(device)
        probs = (torch.sigmoid(model(imgs)) + torch.sigmoid(model(imgs.flip(-1)))) / 2
        all_probs.append(probs.cpu().numpy())

all_probs = np.vstack(all_probs)
all_indices = list(test_ds.indices)
if all_probs.shape[0] != len(all_indices):
    raise ValueError(
        f'Prediction count mismatch: got {all_probs.shape[0]} rows of probabilities '
        f'for {len(all_indices)} test indices.'
    )
preds = (all_probs > thresholds[None, :]).astype(int)

prob_df = pd.DataFrame(all_probs, columns=LABELS, index=all_indices)
prob_df.index.name = 'Id'
pred_df = pd.DataFrame(preds, columns=LABELS, index=all_indices)
pred_df.index.name = 'Id'
prob_df.to_csv(PROBABILITY_CSV)
pred_df.to_csv(BINARY_PREDICTION_CSV)

rows = {'Id': [], 'Predicted': []}
for idx in pred_df.index:
    rows['Id'].append(f'{idx}_classification')
    rows['Predicted'].append(rle_encode(pred_df.loc[idx, LABELS].values.astype(int)))
    rows['Id'].append(f'{idx}_segmentation')
    rows['Predicted'].append('')

submission = pd.DataFrame(rows).set_index('Id')
submission.to_csv(SUBMISSION_CSV)

print('Saved probabilities:', PROBABILITY_CSV)
print('Saved binary predictions:', BINARY_PREDICTION_CSV)
print(f'Saved submission: {SUBMISSION_CSV} ({len(submission)} rows)')
submission.head(4)
